In [ ]:
# Google Colab Setup
# Run this cell first to install dependencies
!pip install -q \
    langchain langchain-core langchain-openai langchain-anthropic \
    langchain-google-genai langchain-google-vertexai langchain-community \
    langchain-tavily langchain-text-splitters langchain-model-profiles \
    langgraph langsmith python-dotenv \
    mcp langchain-mcp-adapters \
    pypdf ipywidgets tqdm tavily


## Without web search

In [ ]:
# API Key Setup
# In Google Colab: open the Secrets panel (🔑 icon in the left sidebar),
# add each key by name, and enable notebook access.
# Locally: keys are loaded from your .env file automatically.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    for key in ["ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "TAVILY_API_KEY",
                "LANGSMITH_API_KEY", "LANGSMITH_TRACING", "LANGSMITH_PROJECT"]:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano"
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
print(response['messages'][-1].content)

## Add web search tool

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
from pprint import pprint

pprint(response['messages'])

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r